# Preliminary Paper Runbook (Colab)

This notebook is the resumable Colab entrypoint for your preliminary paper run.

What it does:
1. Mounts Google Drive and resolves the repo root.
2. Creates a run-specific runtime config under `runs/experiments/<run_id>/configs/`.
3. Uses a stage manifest so completed expensive stages are skipped on rerun.
4. Tracks ETA per stage and shows remaining ETA before and after execution.
5. Produces benchmark, sweep, training, scenario, and backtest artifacts under one run root.

Resume behavior:
- If a stage already completed and its outputs still exist, rerunning the pipeline skips it.
- If a stage fails, fix the issue and rerun the notebook or just rerun the pipeline cell.
- You do not need to rerun completed expensive stages.


In [ ]:
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = None
candidates = [
    Path(os.environ.get("FRM_REPO_DIR", "")).expanduser() if os.environ.get("FRM_REPO_DIR") else None,
    Path.cwd(),
    Path("/content/Forward-Risk-Manager"),
    Path("/content/drive/MyDrive/Forward-Risk-Manager"),
    Path("/content/drive/MyDrive/forward-risk-manager"),
]
for candidate in candidates:
    if candidate is None:
        continue
    if (candidate / "configs" / "default.toml").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Could not locate repo root containing configs/default.toml")

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from frisk.notebook_runtime import (
    NotebookStage,
    format_duration,
    remaining_eta_seconds,
    run_tracked_command,
    shell_quote,
    stage_status_rows,
    step_is_complete,
    write_toml_overrides,
)

print("repo root:", ROOT)
print("cwd:", Path.cwd())


In [ ]:
from datetime import datetime, timezone
import importlib.util
import json

import pandas as pd

PYTHON_EXE = shell_quote(sys.executable)

RUN_ID_OVERRIDE = ""
RESUME_POLICY = "auto"  # "auto" | "force_new" | "force_resume"
DEVICE = "cuda"
INSTALL_DEPS = IN_COLAB

RUN_BUILD_GRAPHS = False
RUN_BENCHMARK = True
RUN_TRAIN = True
RUN_SWEEP = True
RUN_DUAL_SCORE = True
RUN_SCENARIO = True
RUN_BACKTEST = True
RUN_PUBLISH = False

PREBUILT_GRAPH_PATH = ROOT / "data" / "processed" / "graphs_master_ff_rich.pt"

PRICE_CANDIDATES = [
    ROOT / "data" / "processed" / "prices.csv",
    ROOT / "data" / "consolidated_ff_local" / "prices.csv",
    ROOT / "data" / "processed_long" / "prices.csv",
]
CONSTITUENT_CANDIDATES = [
    ROOT / "data" / "processed" / "constituents.csv",
    ROOT / "data" / "processed_long" / "constituents.csv",
]
MACRO_CANDIDATES = [
    ROOT / "data" / "processed" / "macro.csv",
    ROOT / "data" / "consolidated_ff_local" / "macro.csv",
]


def first_existing(paths, *, required=True):
    for path in paths:
        if path.exists():
            return path.resolve()
    if required:
        raise FileNotFoundError("No existing path found in: " + ", ".join(str(p) for p in paths))
    return None


DATA_PRICES_PATH = first_existing(PRICE_CANDIDATES, required=True)
DATA_CONSTITUENTS_PATH = first_existing(CONSTITUENT_CANDIDATES, required=False)
DATA_MACRO_PATH = first_existing(MACRO_CANDIDATES, required=False)

if RESUME_POLICY == "force_resume" and not RUN_ID_OVERRIDE.strip():
    raise ValueError("RESUME_POLICY='force_resume' requires RUN_ID_OVERRIDE")

resume_requested = bool(RUN_ID_OVERRIDE.strip()) and RESUME_POLICY != "force_new"
if resume_requested:
    RUN_ID = RUN_ID_OVERRIDE.strip()
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID
else:
    RUN_ID = f"paper_prelim_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
    RUN_ROOT = ROOT / "runs" / "experiments" / RUN_ID

for sub in ("configs", "data", "metrics", "plots", "logs", "models", "diagnostics"):
    (RUN_ROOT / sub).mkdir(parents=True, exist_ok=True)

RUNTIME_CONFIG = RUN_ROOT / "configs" / "runtime_config.toml"
MANIFEST_PATH = RUN_ROOT / "logs" / "stage_manifest.json"
LOG_DIR = RUN_ROOT / "logs" / "stage_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_OUT = RUN_ROOT / "data" / "graphs_preliminary.pt"
GRAPH_PATH_FOR_RUN = GRAPH_OUT if RUN_BUILD_GRAPHS else PREBUILT_GRAPH_PATH
if not RUN_BUILD_GRAPHS and not GRAPH_PATH_FOR_RUN.exists():
    raise FileNotFoundError(f"Missing prebuilt graph artifact: {GRAPH_PATH_FOR_RUN}")

TRAIN_LOG_CSV = RUN_ROOT / "metrics" / "ff_train.csv"
TRAIN_PLOT = RUN_ROOT / "plots" / "ff_train.png"
MODEL_CKPT = RUN_ROOT / "models" / "ff_model.pt"
ENCODER_CKPT = RUN_ROOT / "models" / "encoder.pt"
CRITIC_CKPT = RUN_ROOT / "models" / "critic.pt"

BENCHMARK_CSV = RUN_ROOT / "metrics" / "benchmark.csv"
BENCHMARK_FOLDS_CSV = RUN_ROOT / "metrics" / "benchmark_walk_forward_folds.csv"
BENCHMARK_BASELINE_CSV = RUN_ROOT / "metrics" / "benchmark_baseline.csv"
BENCHMARK_HISTORY_CSV = RUN_ROOT / "metrics" / "benchmark_history.csv"
BENCHMARK_PLOT = RUN_ROOT / "plots" / "benchmark_speed_sep.png"
BENCHMARK_BAR = RUN_ROOT / "plots" / "benchmark.png"
PAPER_SUMMARY_MD = RUN_ROOT / "logs" / "paper_benchmark_summary.md"
PAPER_SUMMARY_CSV = RUN_ROOT / "metrics" / "paper_benchmark_summary.csv"
PAPER_SUMMARY_JSON = RUN_ROOT / "logs" / "paper_benchmark_summary.json"

SWEEP_CSV = RUN_ROOT / "metrics" / "ff_sweep.csv"
DUAL_SCORE_TXT = RUN_ROOT / "logs" / "dual_score_report.txt"
DUAL_SCORE_CSV = RUN_ROOT / "metrics" / "dual_score_report.csv"

SCENARIO_CSV = RUN_ROOT / "metrics" / "scenario_book.csv"
SCENARIO_DIAG_CSV = RUN_ROOT / "diagnostics" / "scenario_constraint_diagnostics.csv"
STRESS_CSV = RUN_ROOT / "metrics" / "stress_test_report.csv"
STRESS_PLOT = RUN_ROOT / "plots" / "stress_test_report.png"
CALIBRATION_JSON = RUN_ROOT / "diagnostics" / "hallucination_calibration.json"
CALIBRATION_BY_TICKER_CSV = RUN_ROOT / "diagnostics" / "hallucination_calibration_by_ticker.csv"

GOODNESS_CSV = RUN_ROOT / "diagnostics" / "goodness_backtest.csv"
GOODNESS_QUANTILES_CSV = RUN_ROOT / "diagnostics" / "goodness_quantiles.csv"
GOODNESS_PLOT = RUN_ROOT / "plots" / "goodness_scatter.png"
GOODNESS_EVENTS_CSV = RUN_ROOT / "diagnostics" / "goodness_events.csv"
GOODNESS_STRATEGY_CSV = RUN_ROOT / "diagnostics" / "goodness_strategy_metrics.csv"
GOODNESS_TIMELINE_PLOT = RUN_ROOT / "plots" / "goodness_timeline.png"

section_overrides = {
    "build_graphs": {
        "prices": str(DATA_PRICES_PATH),
        "out": str(GRAPH_OUT),
    },
    "train": {
        "graphs": str(GRAPH_PATH_FOR_RUN),
        "device": DEVICE,
        "log_csv": str(TRAIN_LOG_CSV),
        "plot_path": str(TRAIN_PLOT),
        "save_model": str(MODEL_CKPT),
        "save_encoder": str(ENCODER_CKPT),
        "save_critic": str(CRITIC_CKPT),
    },
    "benchmark": {
        "out_csv": str(BENCHMARK_CSV),
        "walk_forward_out_csv": str(BENCHMARK_FOLDS_CSV),
        "baseline_out_csv": str(BENCHMARK_BASELINE_CSV),
        "history_out_csv": str(BENCHMARK_HISTORY_CSV),
        "plot_path": str(BENCHMARK_PLOT),
        "bar_plot_path": str(BENCHMARK_BAR),
        "econ_prices": str(DATA_PRICES_PATH),
    },
    "sweep": {
        "out_csv": str(SWEEP_CSV),
        "econ_prices": str(DATA_PRICES_PATH),
    },
}
if DATA_CONSTITUENTS_PATH is not None:
    section_overrides["build_graphs"]["constituents"] = str(DATA_CONSTITUENTS_PATH)
if DATA_MACRO_PATH is not None:
    section_overrides["build_graphs"]["macro"] = str(DATA_MACRO_PATH)

write_toml_overrides(ROOT / "configs" / "default.toml", RUNTIME_CONFIG, section_overrides)

ETA_MIN = {
    "install": 8,
    "build_graphs": 60,
    "benchmark": 150,
    "paper_summary": 5,
    "train": 180,
    "sweep": 240,
    "dual_score": 5,
    "scenario": 75,
    "stress": 5,
    "calibration": 3,
    "backtest": 20,
    "publish": 2,
}

STAGES = []
if INSTALL_DEPS:
    STAGES.append(NotebookStage("install", "Install dependencies", eta_s=ETA_MIN["install"] * 60))
if RUN_BUILD_GRAPHS:
    STAGES.append(NotebookStage("build_graphs", "Build graphs", (str(GRAPH_OUT),), eta_s=ETA_MIN["build_graphs"] * 60))
if RUN_BENCHMARK:
    STAGES.append(NotebookStage("benchmark", "Run benchmark", (str(BENCHMARK_CSV), str(BENCHMARK_HISTORY_CSV)), eta_s=ETA_MIN["benchmark"] * 60))
    STAGES.append(NotebookStage("paper_summary", "Write paper benchmark summary", (str(PAPER_SUMMARY_MD), str(PAPER_SUMMARY_CSV), str(PAPER_SUMMARY_JSON)), eta_s=ETA_MIN["paper_summary"] * 60))
if RUN_TRAIN:
    STAGES.append(NotebookStage("train", "Train FF/BP model", (str(ENCODER_CKPT), str(CRITIC_CKPT)), eta_s=ETA_MIN["train"] * 60))
if RUN_SWEEP:
    STAGES.append(NotebookStage("sweep", "Run FF sweep", (str(SWEEP_CSV),), eta_s=ETA_MIN["sweep"] * 60))
if RUN_DUAL_SCORE and RUN_BENCHMARK and RUN_SWEEP:
    STAGES.append(NotebookStage("dual_score", "Write dual score report", (str(DUAL_SCORE_TXT), str(DUAL_SCORE_CSV)), eta_s=ETA_MIN["dual_score"] * 60))
if RUN_SCENARIO and RUN_TRAIN:
    STAGES.append(NotebookStage("scenario", "Run scenario book", (str(SCENARIO_CSV), str(SCENARIO_DIAG_CSV)), eta_s=ETA_MIN["scenario"] * 60, optional=True))
    STAGES.append(NotebookStage("stress", "Write stress report", (str(STRESS_CSV), str(STRESS_PLOT)), eta_s=ETA_MIN["stress"] * 60, optional=True))
    STAGES.append(NotebookStage("calibration", "Write hallucination calibration", (str(CALIBRATION_JSON), str(CALIBRATION_BY_TICKER_CSV)), eta_s=ETA_MIN["calibration"] * 60, optional=True))
if RUN_BACKTEST and RUN_TRAIN:
    STAGES.append(NotebookStage("backtest", "Run goodness backtest", (str(GOODNESS_CSV), str(GOODNESS_STRATEGY_CSV), str(GOODNESS_TIMELINE_PLOT)), eta_s=ETA_MIN["backtest"] * 60, optional=True))
if RUN_PUBLISH:
    STAGES.append(NotebookStage("publish", "Publish curated artifacts", eta_s=ETA_MIN["publish"] * 60, optional=True))


def show_stage_status():
    rows = stage_status_rows(MANIFEST_PATH, STAGES, root=ROOT)
    df = pd.DataFrame(rows)
    if not df.empty:
        display(df)
    print("Remaining ETA:", format_duration(remaining_eta_seconds(MANIFEST_PATH, STAGES, root=ROOT)))


print("run id:", RUN_ID)
print("run root:", RUN_ROOT)
print("runtime config:", RUNTIME_CONFIG)
print("graph source:", GRAPH_PATH_FOR_RUN)
print("prices path:", DATA_PRICES_PATH)
show_stage_status()


In [ ]:
required_modules = ["torch", "torch_geometric", "pandas", "numpy", "tqdm", "matplotlib"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if INSTALL_DEPS or missing:
    install_cmd = (
        f"{PYTHON_EXE} -m pip install --upgrade pip setuptools wheel && "
        f"{PYTHON_EXE} -m pip install -r requirements.txt && "
        f"{PYTHON_EXE} -m pip install -e ."
    )
    run_tracked_command(
        step="install",
        label="Install dependencies",
        command=install_cmd,
        manifest_path=MANIFEST_PATH,
        eta_s=ETA_MIN["install"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        metadata={"missing_modules": missing},
    )
else:
    print("Dependencies already installed. Skipping install stage.")

show_stage_status()


In [ ]:
if RUN_BUILD_GRAPHS:
    run_tracked_command(
        step="build_graphs",
        label="Build graphs",
        command=f"{PYTHON_EXE} scripts/build_graphs.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[GRAPH_OUT],
        eta_s=ETA_MIN["build_graphs"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )
else:
    if not GRAPH_PATH_FOR_RUN.exists():
        raise FileNotFoundError(f"Missing graph artifact: {GRAPH_PATH_FOR_RUN}")
    print("Using existing graph artifact:", GRAPH_PATH_FOR_RUN)

if RUN_BENCHMARK:
    run_tracked_command(
        step="benchmark",
        label="Run benchmark",
        command=f"{PYTHON_EXE} scripts/benchmark_training.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[BENCHMARK_CSV, BENCHMARK_HISTORY_CSV, BENCHMARK_PLOT, BENCHMARK_BAR],
        eta_s=ETA_MIN["benchmark"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )
    run_tracked_command(
        step="paper_summary",
        label="Write paper benchmark summary",
        command=(
            f"{PYTHON_EXE} scripts/paper_benchmark_summary.py "
            f"--benchmark {shell_quote(str(BENCHMARK_CSV))} "
            f"--folds-csv {shell_quote(str(BENCHMARK_FOLDS_CSV))} "
            f"--out-md {shell_quote(str(PAPER_SUMMARY_MD))} "
            f"--out-csv {shell_quote(str(PAPER_SUMMARY_CSV))} "
            f"--out-json {shell_quote(str(PAPER_SUMMARY_JSON))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[PAPER_SUMMARY_MD, PAPER_SUMMARY_CSV, PAPER_SUMMARY_JSON],
        eta_s=ETA_MIN["paper_summary"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )

if RUN_TRAIN:
    run_tracked_command(
        step="train",
        label="Train FF/BP model",
        command=f"{PYTHON_EXE} scripts/train_ff_gnn.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[MODEL_CKPT, ENCODER_CKPT, CRITIC_CKPT, TRAIN_LOG_CSV],
        eta_s=ETA_MIN["train"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )

if RUN_SWEEP:
    run_tracked_command(
        step="sweep",
        label="Run FF sweep",
        command=f"{PYTHON_EXE} scripts/ff_sweep.py --config {shell_quote(str(RUNTIME_CONFIG))}",
        manifest_path=MANIFEST_PATH,
        required_outputs=[SWEEP_CSV],
        eta_s=ETA_MIN["sweep"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )

if RUN_DUAL_SCORE and RUN_BENCHMARK and RUN_SWEEP:
    run_tracked_command(
        step="dual_score",
        label="Write dual score report",
        command=(
            f"{PYTHON_EXE} scripts/dual_score_report.py "
            f"--benchmark {shell_quote(str(BENCHMARK_CSV))} "
            f"--sweep {shell_quote(str(SWEEP_CSV))} "
            f"--out {shell_quote(str(DUAL_SCORE_TXT))} "
            f"--out-csv {shell_quote(str(DUAL_SCORE_CSV))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[DUAL_SCORE_TXT, DUAL_SCORE_CSV],
        eta_s=ETA_MIN["dual_score"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
    )

if RUN_SCENARIO and RUN_TRAIN:
    run_tracked_command(
        step="scenario",
        label="Run scenario book",
        command=(
            f"{PYTHON_EXE} scripts/scenario_book.py "
            f"--config {shell_quote(str(RUNTIME_CONFIG))} "
            f"--critic-model {shell_quote(str(CRITIC_CKPT))} "
            f"--out {shell_quote(str(SCENARIO_CSV))} "
            f"--diag-out {shell_quote(str(SCENARIO_DIAG_CSV))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[SCENARIO_CSV, SCENARIO_DIAG_CSV],
        eta_s=ETA_MIN["scenario"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )
    run_tracked_command(
        step="stress",
        label="Write stress report",
        command=(
            f"{PYTHON_EXE} scripts/stress_test_report.py "
            f"--csv {shell_quote(str(SCENARIO_CSV))} "
            f"--out-csv {shell_quote(str(STRESS_CSV))} "
            f"--out-plot {shell_quote(str(STRESS_PLOT))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[STRESS_CSV, STRESS_PLOT],
        eta_s=ETA_MIN["stress"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )
    run_tracked_command(
        step="calibration",
        label="Write hallucination calibration",
        command=(
            f"{PYTHON_EXE} scripts/hallucination_calibration.py "
            f"--csv {shell_quote(str(SCENARIO_CSV))} "
            f"--out {shell_quote(str(CALIBRATION_JSON))} "
            f"--out-by-ticker {shell_quote(str(CALIBRATION_BY_TICKER_CSV))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[CALIBRATION_JSON, CALIBRATION_BY_TICKER_CSV],
        eta_s=ETA_MIN["calibration"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )

if RUN_BACKTEST and RUN_TRAIN:
    run_tracked_command(
        step="backtest",
        label="Run goodness backtest",
        command=(
            f"{PYTHON_EXE} scripts/goodness_backtest.py "
            f"--config {shell_quote(str(RUNTIME_CONFIG))} "
            f"--prices {shell_quote(str(DATA_PRICES_PATH))} "
            f"--out-csv {shell_quote(str(GOODNESS_CSV))} "
            f"--out-quantiles {shell_quote(str(GOODNESS_QUANTILES_CSV))} "
            f"--out-plot {shell_quote(str(GOODNESS_PLOT))} "
            f"--out-events {shell_quote(str(GOODNESS_EVENTS_CSV))} "
            f"--out-strategy {shell_quote(str(GOODNESS_STRATEGY_CSV))} "
            f"--out-timeline {shell_quote(str(GOODNESS_TIMELINE_PLOT))}"
        ),
        manifest_path=MANIFEST_PATH,
        required_outputs=[GOODNESS_CSV, GOODNESS_STRATEGY_CSV, GOODNESS_TIMELINE_PLOT],
        eta_s=ETA_MIN["backtest"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )

if RUN_PUBLISH:
    run_tracked_command(
        step="publish",
        label="Publish curated artifacts",
        command=f"{PYTHON_EXE} scripts/publish_run.py --run-id {shell_quote(RUN_ID)}",
        manifest_path=MANIFEST_PATH,
        eta_s=ETA_MIN["publish"] * 60,
        log_dir=LOG_DIR,
        cwd=ROOT,
        optional=True,
    )

show_stage_status()


In [ ]:
ARTIFACTS = [
    RUNTIME_CONFIG,
    GRAPH_PATH_FOR_RUN,
    TRAIN_LOG_CSV,
    TRAIN_PLOT,
    MODEL_CKPT,
    ENCODER_CKPT,
    CRITIC_CKPT,
    BENCHMARK_CSV,
    BENCHMARK_FOLDS_CSV,
    BENCHMARK_BASELINE_CSV,
    BENCHMARK_HISTORY_CSV,
    BENCHMARK_PLOT,
    BENCHMARK_BAR,
    PAPER_SUMMARY_MD,
    PAPER_SUMMARY_CSV,
    PAPER_SUMMARY_JSON,
    SWEEP_CSV,
    DUAL_SCORE_TXT,
    DUAL_SCORE_CSV,
    SCENARIO_CSV,
    SCENARIO_DIAG_CSV,
    STRESS_CSV,
    STRESS_PLOT,
    CALIBRATION_JSON,
    CALIBRATION_BY_TICKER_CSV,
    GOODNESS_CSV,
    GOODNESS_QUANTILES_CSV,
    GOODNESS_PLOT,
    GOODNESS_EVENTS_CSV,
    GOODNESS_STRATEGY_CSV,
    GOODNESS_TIMELINE_PLOT,
    MANIFEST_PATH,
]

rows = []
for path in ARTIFACTS:
    rows.append({
        "path": str(path),
        "exists": path.exists(),
        "bytes": path.stat().st_size if path.exists() and path.is_file() else None,
    })

display(pd.DataFrame(rows))
show_stage_status()

if PAPER_SUMMARY_MD.exists():
    print("\nPaper summary markdown path:", PAPER_SUMMARY_MD)
if MANIFEST_PATH.exists():
    print("Stage manifest:", MANIFEST_PATH)
    print(MANIFEST_PATH.read_text()[:4000])
